In [1]:
pip install xgboost

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/150.0 MB ? eta -:--:--
   - -------------------------------------- 6.3/150.0 MB 32.2 MB/s eta 0:00:05
   -- ------------------------------------- 11.0/150.0 MB 26.5 MB/s eta 0:00:06
   ---- ----------------------------------- 15.7/150.0 MB 24.7 MB/s eta 0:00:06
   ----- ---------------------------------- 20.2/150.0 MB 24.1 MB/s eta 0:00:06
   ------ --------------------------------- 24.9/150.0 MB 23.5 MB/s eta 0:00:06
   ------- -------------------------------- 29.6/150.0 MB 23.2 MB/s eta 0:00:06
   --------- ------------------------------ 34.1/150.0 MB 23.0 MB/s eta 0:00:06
   ---------- ----------------------------- 39.1/150.0 MB 22.8 MB/s eta 0:00:05
   ----------- ---------------------------- 43.8/150.0 MB 22.8 MB/s eta 0:00:05
   ------------ --------------------------- 48.2/150.0 MB 22.6 MB/s eta 0:00:05
   -------------- ------------------------- 52.7/150

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from datetime import timedelta
import base64
from io import BytesIO
from xgboost import XGBRegressor

# === 1. Load and preprocess data ===
def load_price_data(path):
    df = pd.read_excel(path)
    df.rename(columns={
        'Time from [CET/CEST]': 'timestamp_raw',
        'Price MC Auction [EUR/MWh]': 'price'
    }, inplace=True)

    ts_str = df['timestamp_raw'].astype(str)
    ts_str = ts_str.str.replace(
        r"(\d{4}-\d{2}-\d{2} )(?P<h>\d{1,2})(?:A|B):",
        lambda m: f"{m.group(1)}{int(m.group('h')):02d}:",
        regex=True
    )

    df['timestamp'] = pd.to_datetime(ts_str, errors='coerce')
    df.dropna(subset=['timestamp'], inplace=True)

    df = df.groupby('timestamp')['price'].mean().resample('1h').interpolate()
    df = df.to_frame()  # Convert Series to DataFrame
    df.index = pd.to_datetime(df.index)  # Ensure DatetimeIndex

    return df

# === 2. Setup UI Widgets ===
file_path = r"D:\ML XGboost\Merged_Hourly_DayAhead_2020_2025.xlsx"
data = load_price_data(file_path)

min_date, max_date = data.index.min().date(), data.index.max().date()

from_date = widgets.DatePicker(description='From', value=max_date - timedelta(days=14))
to_date = widgets.DatePicker(description='To', value=max_date)
horizon_radio = widgets.RadioButtons(options=[1, 3, 7], description='Horizon (days)', value=1)
run_button = widgets.Button(description='Run Forecast', button_style='primary')
output = widgets.Output()

# === 3. Download link helper ===
def create_download_link(df, filename="forecast.csv"):
    buffer = BytesIO()
    df.to_csv(buffer, index=False)
    buffer.seek(0)
    b64 = base64.b64encode(buffer.read()).decode()
    return HTML(f'''
        <a download="{filename}" href="data:text/csv;base64,{b64}" target="_blank" style="color:green;font-weight:bold;">
            📥 Download Forecast CSV
        </a>
    ''')

# === 4. Run forecast with ML XGBoost ===
def run_forecast(_):
    with output:
        clear_output()
        start, end = from_date.value, to_date.value
        horizon_days = horizon_radio.value
        forecast_h = horizon_days * 24

        if not start or not end or start >= end:
            print("❌ Please select a valid 'From' and 'To' date.")
            return

        df = data[(data.index.date >= start) & (data.index.date <= end)].copy()
        if len(df) < 500:
            print("⚠️ Not enough data to train XGBoost. Try selecting a longer date range.")
            return

        # === Create features
        df['hour'] = df.index.hour
        df['dayofweek'] = df.index.dayofweek
        df['is_weekend'] = df['dayofweek'].apply(lambda x: 1 if x >= 5 else 0)

        for lag in [1, 2, 24, 48, 168]:
            df[f'lag_{lag}'] = df['price'].shift(lag)

        df = df.dropna()
        X = df.drop(columns='price')
        y = df['price']

        # === Train the model
        model = XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)
        model.fit(X, y)

        # === Forecasting loop
        forecast_points = []
        last_row = df.iloc[-1:].copy()

        for i in range(forecast_h):
            features = last_row.drop(columns='price')
            pred_price = model.predict(features)[0]
            timestamp = last_row.index[0] + timedelta(hours=1)

            new_row = {
                'price': pred_price,
                'hour': timestamp.hour,
                'dayofweek': timestamp.weekday(),
                'is_weekend': 1 if timestamp.weekday() >= 5 else 0,
                'lag_1': pred_price,
                'lag_2': last_row['lag_1'].values[0],
                'lag_24': last_row['lag_24'].values[0],
                'lag_48': last_row['lag_48'].values[0],
                'lag_168': last_row['lag_168'].values[0],
            }

            forecast_points.append((timestamp, pred_price))
            last_row = pd.DataFrame([new_row], index=[timestamp])

        # === Final forecast DataFrame
        forecast_df = pd.DataFrame(forecast_points, columns=['timestamp', 'price'])
        forecast_df['date'] = forecast_df['timestamp'].dt.date
        forecast_df['hour'] = forecast_df['timestamp'].dt.hour

        low_hours = forecast_df.loc[forecast_df.groupby('date')['price'].idxmin()]
        high_hours = forecast_df.loc[forecast_df.groupby('date')['price'].idxmax()]
        res_df = pd.merge(low_hours, high_hours, on='date', suffixes=('_low', '_high'))

        display(res_df[['date', 'hour_low', 'price_low', 'hour_high', 'price_high']])

        # === Plot
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=res_df['date'], y=res_df['hour_low'], mode='lines+markers',
                                 name='Low Hour', marker_color='blue'))
        fig.add_trace(go.Scatter(x=res_df['date'], y=res_df['hour_high'], mode='lines+markers',
                                 name='High Hour', marker_color='orange'))
        fig.add_trace(go.Scatter(x=res_df['date'], y=res_df['price_low'], mode='lines+markers',
                                 name='Low Price €/MWh', yaxis='y2', line=dict(color='blue', dash='dot')))
        fig.add_trace(go.Scatter(x=res_df['date'], y=res_df['price_high'], mode='lines+markers',
                                 name='High Price €/MWh', yaxis='y2', line=dict(color='orange', dash='dot')))

        fig.update_layout(
            title="📈 XGBoost Forecast – Low/High Price Hours",
            xaxis=dict(title="Forecast Date", tickformat="%a %d %b"),
            yaxis=dict(title="Hour of Day", range=[0, 24]),
            yaxis2=dict(title="Price (€/MWh)", overlaying='y', side='right'),
            template="plotly_white", height=500, legend=dict(x=1.02, y=1)
        )
        fig.show()

        display(create_download_link(res_df, filename="xgboost_forecast.csv"))

# === 5. Display Layout ===
run_button.on_click(run_forecast)
ui = widgets.VBox([
    widgets.HBox([from_date, to_date]), horizon_radio, run_button, output
])

display(ui)


In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from datetime import timedelta
import base64
from io import BytesIO
from xgboost import XGBRegressor

# === 1. Load and preprocess data ===
def load_price_data(path):
    df = pd.read_excel(path)
    df.rename(columns={
        'Time from [CET/CEST]': 'timestamp_raw',
        'Price MC Auction [EUR/MWh]': 'price'
    }, inplace=True)

    ts_str = df['timestamp_raw'].astype(str)
    ts_str = ts_str.str.replace(
        r"(\d{4}-\d{2}-\d{2} )(?P<h>\d{1,2})(?:A|B):",
        lambda m: f"{m.group(1)}{int(m.group('h')):02d}:",
        regex=True
    )

    df['timestamp'] = pd.to_datetime(ts_str, errors='coerce')
    df.dropna(subset=['timestamp'], inplace=True)

    df = df.groupby('timestamp')['price'].mean().resample('1h').interpolate()
    df = df.to_frame()
    df.index = pd.to_datetime(df.index)
    return df

# === 2. Setup UI Widgets ===
file_path = r"D:\ML XGboost\Merged_Hourly_DayAhead_2020_2025.xlsx"
data = load_price_data(file_path)

min_date, max_date = data.index.min().date(), data.index.max().date()

from_date = widgets.DatePicker(description='From', value=max_date - timedelta(days=14))
to_date = widgets.DatePicker(description='To', value=max_date)
horizon_radio = widgets.RadioButtons(options=[1, 3, 7], description='Horizon (days)', value=1)
run_button = widgets.Button(description='Run Forecast', button_style='primary')
output = widgets.Output()

# === 3. Download link helper ===
def create_download_link(df, filename="forecast.csv"):
    buffer = BytesIO()
    df.to_csv(buffer, index=False)
    buffer.seek(0)
    b64 = base64.b64encode(buffer.read()).decode()
    return HTML(f'''
        <a download="{filename}" href="data:text/csv;base64,{b64}" target="_blank" style="color:green;font-weight:bold;">
            📥 Download Forecast CSV
        </a>
    ''')

# === 4. Run forecast with ML XGBoost ===
def run_forecast(_):
    with output:
        clear_output()
        start, end = from_date.value, to_date.value
        horizon_days = horizon_radio.value
        forecast_h = horizon_days * 24

        if not start or not end or start >= end:
            print("❌ Please select a valid 'From' and 'To' date.")
            return

        df = data[(data.index.date >= start) & (data.index.date <= end)].copy()
        if len(df) < 500:
            print("⚠️ Not enough data to train XGBoost. Try selecting a longer date range.")
            return

        # === Create features
        df['hour'] = df.index.hour
        df['dayofweek'] = df.index.dayofweek
        df['is_weekend'] = df['dayofweek'].apply(lambda x: 1 if x >= 5 else 0)

        for lag in [1, 2, 24, 48, 168]:
            df[f'lag_{lag}'] = df['price'].shift(lag)

        df = df.dropna()
        X = df.drop(columns='price')
        y = df['price']

        # === Train the model
        model = XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)
        model.fit(X, y)

        # === Forecasting loop
        forecast_points = []
        last_row = df.iloc[-1:].copy()

        for i in range(forecast_h):
            features = last_row.drop(columns='price')
            pred_price = model.predict(features)[0]
            timestamp = last_row.index[0] + timedelta(hours=1)

            new_row = {
                'price': pred_price,
                'hour': timestamp.hour,
                'dayofweek': timestamp.weekday(),
                'is_weekend': 1 if timestamp.weekday() >= 5 else 0,
                'lag_1': pred_price,
                'lag_2': last_row['lag_1'].values[0],
                'lag_24': last_row['lag_24'].values[0],
                'lag_48': last_row['lag_48'].values[0],
                'lag_168': last_row['lag_168'].values[0],
            }

            forecast_points.append((timestamp, pred_price))
            last_row = pd.DataFrame([new_row], index=[timestamp])

        # === Final forecast DataFrame
        forecast_df = pd.DataFrame(forecast_points, columns=['timestamp', 'price'])
        forecast_df['date'] = forecast_df['timestamp'].dt.date
        forecast_df['hour'] = forecast_df['timestamp'].dt.hour

        # === Daily high/low summary
        low_hours = forecast_df.loc[forecast_df.groupby('date')['price'].idxmin()]
        high_hours = forecast_df.loc[forecast_df.groupby('date')['price'].idxmax()]
        res_df = pd.merge(low_hours, high_hours, on='date', suffixes=('_low', '_high'))

        display(res_df[['date', 'hour_low', 'price_low', 'hour_high', 'price_high']])

        # === Plot 1: Low/High Hour Overview
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=res_df['date'], y=res_df['hour_low'], mode='lines+markers',
                                 name='Low Hour', marker_color='blue'))
        fig.add_trace(go.Scatter(x=res_df['date'], y=res_df['hour_high'], mode='lines+markers',
                                 name='High Hour', marker_color='orange'))
        fig.add_trace(go.Scatter(x=res_df['date'], y=res_df['price_low'], mode='lines+markers',
                                 name='Low Price €/MWh', yaxis='y2', line=dict(color='blue', dash='dot')))
        fig.add_trace(go.Scatter(x=res_df['date'], y=res_df['price_high'], mode='lines+markers',
                                 name='High Price €/MWh', yaxis='y2', line=dict(color='orange', dash='dot')))

        fig.update_layout(
            title="📈 XGBoost Forecast – Low/High Price Hours",
            xaxis=dict(title="Forecast Date", tickformat="%a %d %b"),
            yaxis=dict(title="Hour of Day", range=[0, 24]),
            yaxis2=dict(title="Price (€/MWh)", overlaying='y', side='right'),
            template="plotly_white", height=500, legend=dict(x=1.02, y=1)
        )
        fig.show()

        # === Plot 2: Full Hourly Forecast Line (NEW)
        fig_full = go.Figure()
        fig_full.add_trace(go.Scatter(
            x=forecast_df['timestamp'],
            y=forecast_df['price'],
            mode='lines+markers',
            name='Forecasted Price €/MWh',
            line=dict(color='firebrick')
        ))

        fig_full.update_layout(
            title=f"🔮 XGBoost Forecast – Hourly Prices ({horizon_days} day{'s' if horizon_days > 1 else ''})",
            xaxis_title="Timestamp",
            yaxis_title="Price (€/MWh)",
            xaxis=dict(tickformat="%d %b %H:%M"),
            template="plotly_white",
            height=500
        )
        fig_full.show()

        # === CSV Export
        display(create_download_link(res_df, filename="xgboost_forecast.csv"))

# === 5. Display Layout ===
run_button.on_click(run_forecast)
ui = widgets.VBox([
    widgets.HBox([from_date, to_date]), horizon_radio, run_button, output
])

display(ui)

